# 🛡️ Exception Handling & File Processing
---
## 📖 ReadMe
Learn to write **robust, production-ready** Python: handle errors gracefully, work with files, and process common formats (CSV, JSON, text).

| Section | Topics |
|---------|--------|
| Exceptions | try/except/else/finally, hierarchy |
| Custom Exceptions | subclassing Exception |
| Context Managers | `with`, `__enter__`/`__exit__`, `contextlib` |
| File I/O | text, binary, pathlib |
| CSV | csv module, DictReader/DictWriter |
| JSON | json module, serialisation |

> **Level:** Intermediate  
> **Estimated Time:** 90 minutes


## 🧠 Concept Notes

### Exception Hierarchy (selected)
```
BaseException
├── SystemExit
├── KeyboardInterrupt
└── Exception
    ├── ArithmeticError
    │   ├── ZeroDivisionError
    │   └── OverflowError
    ├── LookupError
    │   ├── IndexError
    │   └── KeyError
    ├── TypeError
    ├── ValueError
    ├── IOError / OSError
    │   └── FileNotFoundError
    ├── AttributeError
    ├── ImportError
    └── RuntimeError
```

### try/except anatomy
```python
try:
    # risky code
except SpecificError as e:
    # handle it
except (TypeError, ValueError) as e:
    # handle multiple types
else:
    # runs if NO exception occurred
finally:
    # ALWAYS runs (cleanup)
```

### File Modes
| Mode | Meaning |
|------|---------|
| `r` | Read (default) |
| `w` | Write (truncates) |
| `a` | Append |
| `b` | Binary (add to mode: `rb`, `wb`) |
| `x` | Exclusive create (fails if exists) |
| `+` | Read+Write |


## 🖼️ Diagrams

In [ ]:
print('''
EXCEPTION HANDLING FLOW
═══════════════════════
    try block
        │
        ▼
    exception raised?
        │
    YES │              NO
        │               │
        ▼               ▼
    match except?     else block
        │               │
    YES │ NO            │
        │  │            │
        ▼  ▼            │
    handler re-raise    │
        │               │
        └──────┬────────┘
               ▼
           finally block  ← always runs
               │
               ▼
         continue / exit

CONTEXT MANAGER FLOW
═════════════════════
  with open("file") as f:
        │
        ▼
  __enter__()  ← sets up resource
        │
        ▼
    body runs
        │
        ▼
  __exit__()   ← teardown, even on exception
''')


## ✅ Executable Code

In [ ]:
# ── 1. Basic Exception Handling ──
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Error: Division by zero")
        return None
    except TypeError as e:
        print(f"Type error: {e}")
        return None
    else:
        print(f"{a} / {b} = {result}")
        return result
    finally:
        print("safe_divide: cleanup done")

safe_divide(10, 2)
print()
safe_divide(10, 0)
print()
safe_divide(10, "x")


In [ ]:
# ── 2. Multiple exceptions & re-raising ──
def parse_age(value):
    try:
        age = int(value)
        if age < 0 or age > 150:
            raise ValueError(f"Age {age} out of realistic range")
        return age
    except ValueError as e:
        print(f"Invalid age '{value}': {e}")
        raise   # re-raise so caller knows it failed

for v in ["25", "abc", "-5", "200"]:
    try:
        print(f"Age: {parse_age(v)}")
    except ValueError:
        pass


In [ ]:
# ── 3. Custom Exception Classes ──
class AppError(Exception):
    """Base exception for this application."""

class ValidationError(AppError):
    def __init__(self, field, message):
        self.field = field
        self.message = message
        super().__init__(f"Validation failed on '{field}': {message}")

class DatabaseError(AppError):
    def __init__(self, query, cause=None):
        self.query = query
        super().__init__(f"DB error running: {query}")
        if cause:
            self.__cause__ = cause

# Usage
def validate_email(email):
    if "@" not in email:
        raise ValidationError("email", f"'{email}' is not a valid email")
    return email.lower()

try:
    validate_email("not-an-email")
except ValidationError as e:
    print(f"Field: {e.field}")
    print(f"Error: {e.message}")
except AppError as e:
    print(f"App error: {e}")


In [ ]:
# ── 4. Context Managers ──
import os, tempfile
from contextlib import contextmanager, suppress

# contextmanager decorator — easiest way to make one
@contextmanager
def managed_tempfile(suffix=".txt"):
    """Creates a temp file and deletes it on exit."""
    path = tempfile.mktemp(suffix=suffix)
    print(f"Created temp file: {path}")
    try:
        yield path
    finally:
        if os.path.exists(path):
            os.remove(path)
            print(f"Deleted temp file: {path}")

with managed_tempfile() as tmp:
    with open(tmp, "w") as f:
        f.write("Hello from context manager!")
    with open(tmp) as f:
        print(f.read())

# contextlib.suppress — silence specific errors
with suppress(FileNotFoundError):
    os.remove("/nonexistent/path")
print("Continuing after suppressed error")


In [ ]:
# ── 5. File I/O with pathlib ──
from pathlib import Path
import tempfile

tmp_dir = Path(tempfile.mkdtemp())
sample  = tmp_dir / "sample.txt"

# Write
lines = ["Python is great", "File I/O is easy", "pathlib rocks"]
sample.write_text("\n".join(lines), encoding="utf-8")
print(f"Written {sample.stat().st_size} bytes to {sample.name}")

# Read whole file
content = sample.read_text(encoding="utf-8")
print("Full content:\n", content)

# Read line by line (memory-efficient)
with sample.open(encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        print(f"  Line {i}: {line.strip()}")

# Path operations
print("Stem:", sample.stem)
print("Suffix:", sample.suffix)
print("Parent:", sample.parent)
print("Exists:", sample.exists())

# List directory
for p in tmp_dir.iterdir():
    print("  Found:", p.name)


In [ ]:
# ── 6. CSV Processing ──
import csv
from pathlib import Path
import tempfile

tmp_dir = Path(tempfile.mkdtemp())
csv_file = tmp_dir / "employees.csv"

employees = [
    {"name": "Alice",   "dept": "Engineering", "salary": 95000},
    {"name": "Bob",     "dept": "Marketing",   "salary": 72000},
    {"name": "Carol",   "dept": "Engineering", "salary": 105000},
    {"name": "Dave",    "dept": "Marketing",   "salary": 68000},
    {"name": "Eve",     "dept": "Engineering", "salary": 88000},
]

# Write CSV
with csv_file.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name","dept","salary"])
    writer.writeheader()
    writer.writerows(employees)
print(f"Wrote {len(employees)} rows to {csv_file.name}")

# Read & analyse
dept_totals = {}
with csv_file.open(encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        dept = row["dept"]
        sal  = int(row["salary"])
        dept_totals.setdefault(dept, []).append(sal)

for dept, salaries in dept_totals.items():
    avg = sum(salaries) / len(salaries)
    print(f"  {dept}: avg salary ${avg:,.0f} ({len(salaries)} employees)")


In [ ]:
# ── 7. JSON Processing ──
import json
from pathlib import Path
from datetime import datetime
import tempfile

tmp_dir = Path(tempfile.mkdtemp())
json_file = tmp_dir / "config.json"

config = {
    "app": "DataPipeline",
    "version": "2.1.0",
    "database": {"host": "localhost", "port": 5432, "name": "prod_db"},
    "features": {"cache": True, "async_processing": True, "max_workers": 4},
    "created_at": datetime.now().isoformat()
}

# Serialize to file
with json_file.open("w") as f:
    json.dump(config, f, indent=2)

print(json_file.read_text()[:300], "...")

# Deserialize from file
with json_file.open() as f:
    loaded = json.load(f)

print("\nDB host:", loaded["database"]["host"])
print("Max workers:", loaded["features"]["max_workers"])

# Custom serialiser
class DateTimeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.isoformat()
        return super().default(obj)

data = {"ts": datetime.now(), "value": 42}
print("\nCustom serialised:", json.dumps(data, cls=DateTimeEncoder))


## 📝 Exercises

1. **Safe File Reader** — Write a function that reads a file and returns its lines, handling all possible file errors gracefully with meaningful messages.
2. **Config Validator** — Parse a JSON config file and validate required keys exist with correct types; raise `ValidationError` with details.
3. **CSV Aggregator** — Read a CSV of sales records and compute total, average, min, max by region.
4. **Log Parser** — Parse a multi-line log file and extract ERROR lines with timestamps.
5. **Retry Context Manager** — Build a context manager that retries the block up to n times on failure.


## ✔️ Solutions

In [ ]:
from pathlib import Path
import csv, json, re, tempfile
from contextlib import contextmanager
import time

# 1. Safe File Reader
def safe_read_lines(path):
    try:
        return Path(path).read_text(encoding="utf-8").splitlines()
    except FileNotFoundError:
        print(f"File not found: {path}")
    except PermissionError:
        print(f"No permission to read: {path}")
    except UnicodeDecodeError as e:
        print(f"Encoding error: {e}")
    return []

# 2. Config Validator
REQUIRED = {"app": str, "version": str, "database": dict}

def validate_config(path):
    with open(path) as f:
        cfg = json.load(f)
    errors = []
    for key, expected_type in REQUIRED.items():
        if key not in cfg:
            errors.append(f"Missing required key '{key}'")
        elif not isinstance(cfg[key], expected_type):
            errors.append(f"'{key}' must be {expected_type.__name__}")
    if errors:
        raise ValidationError("config", "; ".join(errors))
    return cfg

# 4. Log Parser
def parse_errors(log_text):
    pattern = re.compile(r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}).*ERROR.*?(.+)$", re.M)
    return [{"timestamp": m.group(1), "message": m.group(2).strip()}
            for m in pattern.finditer(log_text)]

sample_log = """2024-01-15 09:00:00 INFO  Service started
2024-01-15 09:01:00 ERROR Database connection failed: timeout
2024-01-15 09:02:00 INFO  Retrying...
2024-01-15 09:02:05 ERROR Auth service unreachable: connection refused"""

errors = parse_errors(sample_log)
print("Parsed errors:")
for e in errors:
    print(f"  [{e['timestamp']}] {e['message']}")

# 5. Retry Context Manager
@contextmanager
def retry_block(times=3, delay=0.1, exceptions=(Exception,)):
    for attempt in range(1, times + 1):
        try:
            yield attempt
            break
        except exceptions as e:
            print(f"Attempt {attempt}/{times} failed: {e}")
            if attempt < times:
                time.sleep(delay)
            else:
                raise

import random
random.seed(42)
with retry_block(times=3) as attempt:
    print(f"Attempt {attempt}")
    if attempt < 2:
        raise RuntimeError("Simulated failure")
    print("Success!")


## 💼 Enterprise Examples

In [ ]:
# Enterprise: Resilient ETL pipeline with error handling
import csv, json, logging
from pathlib import Path
from contextlib import contextmanager
from collections import defaultdict
import tempfile

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("etl")

class ETLError(Exception): pass
class TransformError(ETLError):
    def __init__(self, row, reason):
        self.row = row
        super().__init__(f"Transform failed on row {row}: {reason}")

def transform_record(raw):
    """Validate and transform a raw CSV row."""
    try:
        return {
            "id":       int(raw["id"]),
            "name":     raw["name"].strip().title(),
            "revenue":  float(raw["revenue"].replace("$","").replace(",","")),
            "region":   raw["region"].upper(),
        }
    except (KeyError, ValueError) as e:
        raise TransformError(raw, str(e)) from e

def etl_pipeline(source_csv):
    """Full ETL: extract → transform → load (to JSON + stats)."""
    records, errors = [], []

    with open(source_csv, newline="", encoding="utf-8") as f:
        for i, row in enumerate(csv.DictReader(f), 1):
            try:
                records.append(transform_record(row))
            except TransformError as e:
                log.warning(f"Row {i} skipped: {e}")
                errors.append({"row": i, "error": str(e)})

    # Aggregate
    stats = defaultdict(lambda: {"count":0,"total":0.0})
    for r in records:
        s = stats[r["region"]]
        s["count"] += 1
        s["total"] += r["revenue"]

    log.info(f"Processed {len(records)} records, {len(errors)} errors")
    return records, dict(stats), errors

# Generate sample CSV
tmp = Path(tempfile.mkdtemp())
raw_csv = tmp / "sales.csv"
raw_csv.write_text(
    "id,name,revenue,region\n"
    "1,alice smith,$1,250.50,north\n"
    "2,bob jones,$2,100.00,south\n"
    "3,INVALID,notanumber,east\n"
    "4,carol white,$3,400.75,north\n"
)

records, stats, errors = etl_pipeline(raw_csv)
print("\nRecords:", records)
print("\nStats:")
for region, s in stats.items():
    print(f"  {region}: {s['count']} deals, ${s['total']:,.2f} total")
print("\nErrors:", errors)


## ❓ Interview Questions

**Q1: What is the difference between `except Exception` and `except BaseException`?**
> `BaseException` also catches `SystemExit`, `KeyboardInterrupt`, and `GeneratorExit` — things you usually *don't* want to catch. Always use `except Exception` unless you specifically need the others.

**Q2: When does `finally` NOT run?**
> If the process is killed with `SIGKILL`, `os._exit()`, or a hardware failure. It always runs on normal exceptions and `SystemExit`.

**Q3: What's the purpose of the `else` clause in try/except?**
> It runs only when **no exception** occurred in the `try` block. Useful to separate "normal" code from error-handling code.

**Q4: Why use `with open(...)` instead of `f = open(...)`?**
> `with` guarantees `f.close()` is called even if an exception occurs — prevents file descriptor leaks.

**Q5: How do you chain exceptions?**
> Use `raise NewError("msg") from original_error` to attach the original as `__cause__`, preserving the full traceback chain.


## ⚠️ Common Mistakes

| Mistake | Problem | Fix |
|---------|---------|-----|
| Bare `except:` | Catches everything including SystemExit | `except Exception as e:` |
| Swallowing exceptions | `except: pass` hides bugs | At minimum `log.exception(e)` |
| Not closing files | File descriptor leak | Use `with open(...)` |
| Catching broad exception | Masks real bugs | Catch the most specific type |
| `raise e` vs `raise` | `raise e` resets traceback | `raise` re-raises with original traceback |
| Opening in wrong mode | `'w'` truncates existing file | Use `'a'` for append, `'x'` for exclusive create |


## 📄 Cheat Sheet
```python
# TRY / EXCEPT
try:
    ...
except (TypeError, ValueError) as e:
    ...
else:
    ...      # no exception
finally:
    ...      # always

# RE-RAISE
raise             # re-raise current
raise e           # raise (resets traceback)
raise NewError() from e   # chain

# CUSTOM EXCEPTION
class MyError(Exception):
    def __init__(self, msg, code=None):
        super().__init__(msg)
        self.code = code

# CONTEXT MANAGER
with open("file.txt", "r", encoding="utf-8") as f:
    content = f.read()

@contextmanager
def my_cm():
    # setup
    try:
        yield resource
    finally:
        # teardown

# PATHLIB
p = Path("dir/file.txt")
p.read_text()  p.write_text(s)  p.exists()
p.parent  p.stem  p.suffix  p.name
p.mkdir(parents=True, exist_ok=True)
list(p.glob("*.csv"))

# CSV
with open("f.csv", newline="") as f:
    for row in csv.DictReader(f): ...

# JSON
json.dumps(obj, indent=2)   json.loads(s)
json.dump(obj, file)        json.load(file)
```
